In [ ]:
# This is done because the code correction done to extract features file to include patch_size_level0 was not done correctly. We can correct the code in that file if we ever run this again 
# AND SKIP THIS POST PROCESSING IN THAT CASE


import h5py
import glob
import os
import shutil
from pathlib import Path

def save_hdf5(output_path, asset_dict, attr_dict=None, mode='a', chunk_size=32):
    """Copy of CLAM's save_hdf5 function"""
    with h5py.File(output_path, mode) as file:
        for key, val in asset_dict.items():
            data_shape = val.shape
            if key not in file:
                data_type = val.dtype
                chunk_shape = (chunk_size, ) + data_shape[1:]
                maxshape = (None, ) + data_shape[1:]
                dset = file.create_dataset(key, shape=data_shape, maxshape=maxshape, chunks=chunk_shape, dtype=data_type)
                dset[:] = val
                if attr_dict is not None:
                    if key in attr_dict.keys():
                        for attr_key, attr_val in attr_dict[key].items():
                            dset.attrs[attr_key] = attr_val
            else:
                dset = file[key]
                dset.resize(len(dset) + data_shape[0], axis=0)
                dset[-data_shape[0]:] = val
    return output_path

def rescue_conch_features_safe():
    """
    NON-DESTRUCTIVE: Create corrected copies in new directory
    Your original files remain completely untouched!
    """
    # Paths
    features_dir = "/home/woody/iwi5/iwi5204h/CLAM/CONCH_1_5_features"
    source_dir = os.path.join(features_dir, "h5_files")  # Your original files
    rescue_dir = os.path.join(features_dir, "h5_files_postprocessed")  # New corrected files
    
    # Create rescue directory
    Path(rescue_dir).mkdir(parents=True, exist_ok=True)
    
    # Find all h5 files
    h5_files = glob.glob(os.path.join(source_dir, "*.h5"))
    
    print(f"Creating TITAN-compatible copies of {len(h5_files)} files...")
    print(f"Source: {source_dir}")
    print(f"Destination: {rescue_dir}")
    print(f"Original files to remain untouched!\n")
    
    success_count = 0
    failed_files = []
    
    for source_file in h5_files:
        try:
            # Define paths
            filename = os.path.basename(source_file)
            destination_file = os.path.join(rescue_dir, filename)
            
            # Read data from original file
            with h5py.File(source_file, 'r') as f_source:
                features = f_source['features'][:]
                coords = f_source['coords'][:]
                
                # Check if already has correct attributes (skip if already good)
                if 'patch_size_level0' in f_source['coords'].attrs:
                    print(f"{filename} already has patch_size_level0, skipping...")
                    continue
            
            # Create new file with correct structure
            asset_dict = {'features': features, 'coords': coords}
            attr_dict = {
                'coords': {
                    'patch_size_level0': 512,  # REG2025 at 20x magnification
                    'patch_level': 0
                }
            }
            
            # Write to new location with proper attributes
            save_hdf5(destination_file, asset_dict, attr_dict=attr_dict, mode='w')
            
            # Verify the new file
            with h5py.File(destination_file, 'r') as f_verify:
                coords_attrs = dict(f_verify['coords'].attrs)
                
                # Verify attributes
                assert 'patch_size_level0' in coords_attrs, "patch_size_level0 missing"
                assert coords_attrs['patch_size_level0'] == 512, f"Wrong patch_size_level0: {coords_attrs['patch_size_level0']}"
                
                # Verify data integrity
                assert f_verify['features'].shape == features.shape, "Features shape mismatch"
                assert f_verify['coords'].shape == coords.shape, "Coords shape mismatch"
                
                # Verify data content (sample check)
                assert f_verify['features'][0, 0] == features[0, 0], "Features data mismatch"
            
            print(f"Created: {filename} ({features.shape[0]} patches)")
            success_count += 1
            
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            failed_files.append(filename)
            # Remove failed file if it was partially created
            if os.path.exists(destination_file):
                os.remove(destination_file)
    
    # Summary
    print(f"\n🎉 POST-PROCESSING COMPLETE!")
    print(f"Successfully processed: {success_count}/{len(h5_files)} files")
    print(f"Post-processed files location: {rescue_dir}")
    
    if failed_files:
        print(f"Failed files: {failed_files}")
    
    # Test TITAN compatibility on one file
    if success_count > 0:
        test_titan_compatibility(rescue_dir)
    
    return success_count, failed_files

def test_titan_compatibility(rescue_dir):
    """Test if rescued files work with TITAN format expectations"""
    print(f"\nTesting TITAN compatibility...")
    
    # Get first rescued file
    rescued_files = glob.glob(os.path.join(rescue_dir, "*.h5"))
    if not rescued_files:
        return
        
    test_file = rescued_files[0]
    
    try:
        # Simulate TITAN's expected loading pattern
        with h5py.File(test_file, 'r') as file:
            features = file['features'][:]
            coords = file['coords'][:]
            patch_size_lv0 = file['coords'].attrs['patch_size_level0']
            
        print(f"TITAN compatibility test PASSED!")
        print(f"   - Features shape: {features.shape}")
        print(f"   - Coords shape: {coords.shape}")
        print(f"   - patch_size_level0: {patch_size_lv0}")
        
    except Exception as e:
        print(f"TITAN compatibility test FAILED: {e}")

def compare_file_sizes():
    """Compare original vs rescued file sizes"""
    features_dir = "/home/woody/iwi5/iwi5204h/CLAM/CONCH_1_5_features"
    source_dir = os.path.join(features_dir, "h5_files_back")
    rescue_dir = os.path.join(features_dir, "h5_files_rescued")
    
    print(f"\nFile size comparison:")
    
    for source_file in glob.glob(os.path.join(source_dir, "*.h5"))[:3]:  # Check first 3
        filename = os.path.basename(source_file)
        rescue_file = os.path.join(rescue_dir, filename)
        
        if os.path.exists(rescue_file):
            source_size = os.path.getsize(source_file) / (1024*1024)  # MB
            rescue_size = os.path.getsize(rescue_file) / (1024*1024)  # MB
            print(f"   {filename}: {source_size:.1f}MB → {rescue_size:.1f}MB")

if __name__ == "__main__":
    # Run the safe rescue
    success_count, failed_files = rescue_conch_features_safe()
    
    # Compare sizes (optional)
    if success_count > 0:
        compare_file_sizes()
    
    print(f"\nNext steps:")
    print(f"   1. Test TITAN with files in h5_files_rescued/")
    print(f"   2. If successful, you can safely delete h5_files_back/")
    print(f"   3. Rename h5_files_rescued/ to h5_files/")


Creating TITAN-compatible copies of 8352 files...
Source: /home/woody/iwi5/iwi5204h/CLAM/CONCH_1_5_features/h5_files
Destination: /home/woody/iwi5/iwi5204h/CLAM/CONCH_1_5_features/h5_files_postprocessed
Original files to remain untouched!

Created: PIT_02_00134_01.h5 (21173 patches)
Created: PIT_01_10262_01.h5 (2717 patches)
Created: PIT_01_06639_01.h5 (317 patches)
Created: PIT_01_01997_01.h5 (483 patches)
Created: PIT_01_07085_01.h5 (594 patches)
Created: PIT_01_06473_01.h5 (142 patches)
Created: PIT_01_10028_01.h5 (3934 patches)
Created: PIT_01_02599_01.h5 (505 patches)
Created: PIT_01_02136_01.h5 (630 patches)
Created: PIT_01_10487_01.h5 (1756 patches)
Created: PIT_01_07660_01.h5 (134 patches)
Created: PIT_01_00510_01.h5 (156 patches)
Created: PIT_01_08344_01.h5 (229 patches)
Created: PIT_01_06588_02.h5 (240 patches)
Created: PIT_01_01549_01.h5 (287 patches)
Created: PIT_01_01703_01.h5 (384 patches)
Created: PIT_01_02947_01.h5 (431 patches)
Created: PIT_01_01840_01.h5 (309 patches)